# Chapter 9 — 데이터 전처리와 Feature 생성 (실습)

## 실습 목표
1. inf/NaN 처리 (replace + KNNImputer)
2. RobustScaler 정규화
3. 명목형 One-Hot 인코딩
4. 시간·도메인 파생 feature 5개 생성
5. sklearn Pipeline 으로 묶기

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib
# !pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(42)

In [ ]:
# 데이터 준비 (ch08 와 유사한 mock)
def make_rows(n, label, dur, pkts, syn, ports):
    return pd.DataFrame({
        'Source IP': rng.choice([f'10.0.0.{i}' for i in range(1,50)], size=n),
        'Destination Port': rng.choice(ports, size=n),
        'Flow Duration': rng.uniform(*dur, size=n),
        'Total Fwd Packets': rng.integers(*pkts, size=n),
        'Total Backward Packets': rng.integers(0, 10, size=n) if label != 'BENIGN' else rng.integers(*pkts, size=n),
        'Total Length of Fwd Packets': rng.integers(*pkts, size=n) * rng.integers(60, 1500, size=n),
        'SYN Flag Count': rng.uniform(*syn, size=n),
        'ACK Flag Count': rng.uniform(0.0 if label != 'BENIGN' else 0.4, 0.6, size=n),
        'Protocol': rng.choice(['TCP', 'UDP'], size=n, p=[0.85, 0.15]),
        'Timestamp': pd.to_datetime('2024-07-05') + pd.to_timedelta(rng.integers(0, 86400, size=n), unit='s'),
        'Label': label,
    })

df = pd.concat([
    make_rows(16000, 'BENIGN',   (1000, 1_000_000), (5, 100),    (0, 0.05),  [80, 443, 53, 22]),
    make_rows( 2300, 'DDoS',     (10, 5000),         (500, 5000), (0.4, 0.7), [80, 443]),
    make_rows( 1200, 'PortScan', (1, 200),           (1, 5),      (0.85, 1.0), list(range(1, 1024))),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

# inf 시뮬레이션
df['Flow Bytes/s'] = np.where(df['Flow Duration'] < 0.001, np.inf, df['Total Length of Fwd Packets'] / df['Flow Duration'].clip(lower=1e-6))
# 결측 시뮬레이션 (Flow Duration 의 1% 를 NaN)
mask = rng.random(len(df)) < 0.01
df.loc[mask, 'Flow Duration'] = np.nan
print(f'shape: {df.shape}')
df.head()

---
## Step 1. inf/NaN 처리

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. inf/NaN 처리
# 만들 결과: df, miss_pct, num_cols, imp, df[num_cols]
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 1) inf → NaN
df = ...
miss_pct = ...
...

# 2) 먼저 분할하고 학습 행에서만 대치기 학습
from sklearn.model_selection import train_test_split
train_idx, test_idx = ...
num_cols = ...
            ...
imp = ...
...
df[num_cols] = ...
...
```
</details>


---
## Step 2. 이상치 — clipping + log

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 이상치 — clipping + log
# 만들 결과: upper, df[col], df['log_total_bytes'], df['log_flow_dur']
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# Winsorize 99 percentile
for col in ['Flow Bytes/s', 'Total Length of Fwd Packets']:
    upper = ...
    df[col] = ...

# log 변환 — 우측 치우친 분포
df['log_total_bytes'] = ...
df['log_flow_dur'] = ...

fig, axes = ...
axes[0].hist(df['Total Length of Fwd Packets'], bins = ...
...
axes[1].hist(df['log_total_bytes'], bins = ...
...
```
</details>


---
## Step 3. 인코딩 — One-Hot + 라벨 인코딩

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 인코딩 — One-Hot + 라벨 인코딩
# 만들 결과: df, df['label_bin'], le, df['label_enc']
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# Protocol One-Hot
df = ...

# 라벨 인코딩 — 이진 + 다중
df['label_bin'] = ...
le = ...
df['label_enc'] = ...
...
```
</details>


---
## Step 4. 파생 Feature 5개

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 파생 Feature 5개
# 만들 결과: ts, df['hour'], df['is_night'], df['bytes_per_pkt'], df['bwd_to_fwd_ratio'], df['syn_dominance']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
ts = ...
df['hour'] = ...
df['is_night'] = ...
df['bytes_per_pkt'] = ...
df['bwd_to_fwd_ratio'] = ...
df['syn_dominance'] = ...
df['is_well_known_port'] = ...

# 라벨별 파생 feature 평균 비교
derived = ...
...
```
</details>


관찰: `syn_dominance` 와 `is_well_known_port` 는 라벨별로 명확히 다르다 → 강한 feature.

---
## Step 5. sklearn Pipeline 으로 통합

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. sklearn Pipeline 으로 통합
# 만들 결과: feature_cols, X, y, pipe, X_train_proc, X_test_proc
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
from sklearn.model_selection import train_test_split

feature_cols = ...
                ...
X = ...
y = ...

X_train, X_test = ...
y_train, y_test = ...

pipe = ...
    ('impute', KNNImputer(n_neighbors = ...
    ...
X_train_proc = ...
X_test_proc = ...
print(f'X_train_proc.shape = ...
print(f'평균(첫 3 feature): {X_train_proc[:, :3].mean(axis = ...
print(f'중앙(첫 3 feature): {np.median(X_train_proc[:, :3], axis = ...
```
</details>


---
## 정리
- KISIA Part 3 의 IRIS 전처리 패턴이 보안 데이터에 그대로 적용된다.
- 보안에서는 RobustScaler + clipping + log 변환이 표준.
- 파생 feature (syn_dominance, is_well_known_port) 가 라벨 구분에 강하다.
- Pipeline 으로 묶으면 fit/transform 일관성을 자동 보장 — leakage 방지.
- 다음 챕터에서 이 전처리 결과로 EDA 와 시각화를 본격 한다.